# 01 — Exploratory Data Analysis

This notebook explores the ESC-50 training dataset:
- Class distribution
- Waveform visualisation
- Spectrogram visualisation
- Effect of preprocessing on the signal
- Effect of each augmentation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

import sys
sys.path.insert(0, '..')   # allow imports from project root

from src.preprocessing import load_audio, preprocess_audio
from src.augmentation import augment_noise, augment_gain, bandpass_filter, augment_shift

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT   = os.environ.get('DATA_ROOT', '../data')
TRAIN_DIR   = os.path.join(DATA_ROOT, 'train')
AUDIO_DIR   = os.path.join(TRAIN_DIR, 'audio')
LABELS_CSV  = os.path.join(TRAIN_DIR, 'labels.csv')

df = pd.read_csv(LABELS_CSV)
df['clip_id'] = df['clip_id'].astype(str)
print(f'{len(df)} clips | {df["label"].nunique()} classes')
df.head()

## 1. Class Distribution

In [ ]:
counts = df['label'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.axhline(counts.mean(), color='red', linestyle='--', label=f'Mean = {counts.mean():.1f}')
ax.set_xlabel('Sound Class')
ax.set_ylabel('Number of Clips')
ax.set_title('Class Distribution — Training Set')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=90, fontsize=8)
ax.legend()
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Clips per class:', counts.unique())

## 2. Waveform & Spectrogram of Sample Clips

Pick one clip from each of a few interesting classes.

In [ ]:
SAMPLE_CLASSES = ['dog', 'rain', 'car_horn', 'chainsaw', 'helicopter']
SR = 16000

fig, axes = plt.subplots(len(SAMPLE_CLASSES), 2, figsize=(14, 3 * len(SAMPLE_CLASSES)))

for row_idx, cls in enumerate(SAMPLE_CLASSES):
    clip_id = df[df['label'] == cls].iloc[0]['clip_id']
    path = os.path.join(AUDIO_DIR, f'{clip_id}.wav')
    y, sr = load_audio(path, SR)
    y_proc = preprocess_audio(y, sr)

    # Waveform
    ax_w = axes[row_idx, 0]
    t = np.linspace(0, len(y_proc) / sr, len(y_proc))
    ax_w.plot(t, y_proc, linewidth=0.5, color='steelblue')
    ax_w.set_title(f'{cls} — Waveform')
    ax_w.set_xlabel('Time (s)')
    ax_w.set_ylabel('Amplitude')

    # Mel Spectrogram
    ax_s = axes[row_idx, 1]
    mel = librosa.feature.melspectrogram(y=y_proc, sr=sr, n_mels=128)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(log_mel, sr=sr, hop_length=512, x_axis='time',
                                    y_axis='mel', ax=ax_s)
    ax_s.set_title(f'{cls} — Log-Mel Spectrogram')
    fig.colorbar(img, ax=ax_s, format='%+2.0f dB')

plt.tight_layout()
plt.savefig('waveforms_spectrograms.png', dpi=150)
plt.show()

## 3. Effect of Preprocessing

Compare raw vs preprocessed signal for the same clip.

In [ ]:
DEMO_CLASS = 'dog'
clip_id = df[df['label'] == DEMO_CLASS].iloc[0]['clip_id']
path = os.path.join(AUDIO_DIR, f'{clip_id}.wav')

y_raw, sr = load_audio(path)
y_proc    = preprocess_audio(y_raw, sr)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
for ax, sig, title in zip(axes, [y_raw, y_proc], ['Raw', 'Preprocessed']):
    t = np.linspace(0, len(sig) / sr, len(sig))
    ax.plot(t, sig, linewidth=0.5)
    ax.set_title(f'{DEMO_CLASS} — {title}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
plt.tight_layout()
plt.savefig('preprocessing_comparison.png', dpi=150)
plt.show()

## 4. Visualising Augmentations

Show waveforms and spectrograms for each augmentation type on one clip.

In [ ]:
augmentations = {
    'Original'  : y_proc,
    'Noise'     : augment_noise(y_proc),
    'Gain'      : augment_gain(y_proc),
    'Bandpass'  : bandpass_filter(y_proc, sr),
    'Shift'     : augment_shift(y_proc),
}

fig, axes = plt.subplots(len(augmentations), 2, figsize=(14, 3 * len(augmentations)))

for idx, (name, sig) in enumerate(augmentations.items()):
    t = np.linspace(0, len(sig) / sr, len(sig))

    axes[idx, 0].plot(t, sig, linewidth=0.5, color='darkorange')
    axes[idx, 0].set_title(f'{name} — Waveform')
    axes[idx, 0].set_xlabel('Time (s)')
    axes[idx, 0].set_ylabel('Amplitude')

    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=sig, sr=sr, n_mels=64), ref=np.max)
    img = librosa.display.specshow(mel, sr=sr, hop_length=512, x_axis='time',
                                    y_axis='mel', ax=axes[idx, 1])
    axes[idx, 1].set_title(f'{name} — Log-Mel Spectrogram')
    fig.colorbar(img, ax=axes[idx, 1], format='%+2.0f dB')

plt.tight_layout()
plt.savefig('augmentation_comparison.png', dpi=150)
plt.show()